In [3]:
!uv add ipykernel

Resolved 69 packages in 789ms
Audited 65 packages in 15ms


In [28]:
from pathlib import Path
import shutil
import time

In [ ]:
PROJECT_ROOT =Path('test.ipynb').resolve().parent
AUTO_SCRIPT_PATH = PROJECT_ROOT / 'automation_mvn_tests'
DEFAULT_TEST_SUITE = Path('automation_mvn_tests\src\test\resources\testng.xml')
ALLOWED_MAVEN_GOALS = {"test", "verify"}


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
C:\Users\yashp\AppData\Local\Temp\ipykernel_25576\3341238168.py:3: SyntaxWarning: invalid escape sequence '\s'
  DEFAULT_TEST_SUITE = Path('automation_mvn_tests\src\test\resources\testng.xml')


In [12]:
def __tail_output(output: str, max_chars: int) -> str:
    if len(output) <= max_chars:
        return output
    else:
        return f"Output truncated: {output}"

In [14]:
def __normalize_stream(stream: str | bytes | None) -> str:
    if stream is None:
        return ""
    if isinstance(stream, bytes):
        return stream.decode(errors= "replace")
    return stream
    

In [19]:
def _find_maven_command() -> list[str] | None:
    print("In the _find_maven_command function")
    wrapper_candidates = [
        AUTO_SCRIPT_PATH / "mvnw.cmd",  # Windows wrapper
        AUTO_SCRIPT_PATH / "mvnw",      # Unix wrapper
        AUTO_SCRIPT_PATH / "mvn.bat",  # Additional Windows wrapper
    ]
    for candidate in wrapper_candidates:
        if candidate.is_file():
            print(f"Candidate found: {candidate}")
            return [str(candidate)]
    
    for executable in ["mvnw.cmd", "mvnw", "mvn.bat"]:
        resolved = shutil.which(executable)
        if resolved:
            print(f"Resolved executable found: {resolved}")
            return [resolved]
        
    return None

In [24]:
f = _find_maven_command()
print(f)

In the _find_maven_command function
None


In [16]:
def _resolve_suite_path(test_suite: str) -> Path:
    suite_rel_path = Path(test_suite.strip() or str(DEFAULT_TEST_SUITE))
    if suite_rel_path.is_absolute():
        raise ValueError("TestNG suite path must be relative under automate test tool project folder")
    
    suite_abs_path = (AUTO_SCRIPT_PATH / suite_rel_path).resolve()
    if not suite_abs_path.is_relative_to(AUTO_SCRIPT_PATH):
        raise ValueError("TestNG Suite should be part of the Automation test script project folder structure")
    
    if not suite_abs_path.is_file():
        raise FileNotFoundError(f"TestNG suite missing: {suite_abs_path.as_posix()}")
    
    return suite_abs_path


In [25]:
f1 = _resolve_suite_path("src/test/resources/testng.xml")
print(f1)

C:\Users\yashp\OneDrive\Documents\Phani\new_mcp\new_api_mcp\automation_mvn_tests\src\test\resources\testng.xml


In [26]:
def _collect_report_paths() -> list[str]:
    report_candidates = [
        AUTO_SCRIPT_PATH / "target" / "surefire-reports" / "testng-results.xml",
        AUTO_SCRIPT_PATH / "target" / "surefire-reports" / "emailable-report.html",
        AUTO_SCRIPT_PATH / "target" / "cucumber.json",
        AUTO_SCRIPT_PATH / "target" / "cucumber-reports",
    ]

    available_reports: list[str] = []
    for candidate in report_candidates:
        if candidate.exists():
            available_reports.append(candidate.relative_to(PROJECT_ROOT).as_posix())
    
    print(f"Available reports: {available_reports}")

    return available_reports


In [27]:
f2 = _collect_report_paths()
print(f2)

Available reports: ['automation_mvn_tests/target/surefire-reports/testng-results.xml', 'automation_mvn_tests/target/surefire-reports/emailable-report.html', 'automation_mvn_tests/target/cucumber.json', 'automation_mvn_tests/target/cucumber-reports']
['automation_mvn_tests/target/surefire-reports/testng-results.xml', 'automation_mvn_tests/target/surefire-reports/emailable-report.html', 'automation_mvn_tests/target/cucumber.json', 'automation_mvn_tests/target/cucumber-reports']


In [30]:
import json

def run_auto_script(
        maven_goal: str = "test",
        clean_first: bool = False,
        testng_suite: str = "src/test/resources/testng.xml",
        cucumber_tags: str = "",
        timeout_in_sec: int = 900,
        max_output_chars: int = 8000
) -> str:
    """Run Maven automation scripts and return structured outputs"""

    if not AUTO_SCRIPT_PATH.is_dir():
        return json.dumps({
            "status": "error",
            "error": "Automation script directory does not exist",
            "working_directory": AUTO_SCRIPT_PATH.as_posix()
        }, indent=2)
    
    goal = maven_goal.strip().lower()
    if goal not in ALLOWED_MAVEN_GOALS:
        return json.dumps({
            "status": "error",
            "error": f"Unsupported Maven goal '{maven_goal}'. Allowed goals: {', '.join(ALLOWED_MAVEN_GOALS)}"
        }, indent=2)
    
    timeout_sec = max(60, min(timeout_in_sec, 3600)) # enforce 1 min to 1 hour range
    max_output_chars = max(1000, min(max_output_chars, 20000)) # enforce 1k to 20k chars

    try:
       suite_abs_path =  _resolve_suite_path(testng_suite)
       print(f"Resolved TestNG suite path: {suite_abs_path}")
    except (ValueError, FileNotFoundError) as e:
        return json.dumps({
                "status": "error",
                "error": str(e)
        })
    
    maven_command = _find_maven_command()
    # print(f"Resolved Maven command: {maven_command}")
    if not maven_command:
        return json.dumps({
            "status": "error",
            "error": "Maven command not found. Ensure Maven is installed and/or mvnw wrapper is present in the automation script directory."
        }, indent=2)
    
    command = [*maven_command]
    # print(command)
    if clean_first:
        command.append("clean")
    command.append(goal)

    suite_rel_path = suite_abs_path.relative_to(AUTO_SCRIPT_PATH).as_posix()
    command.append(f"-Dsurefire.suiteXmlFiles={suite_rel_path}")

    if cucumber_tags.strip():
        command.append(f"-Dcucumber.filter.tags={cucumber_tags.strip()}")
    print(f"Final command to run: {command}")

    start = time.perf_counter()
    try:
        import subprocess, asyncio
        result = asyncio.to_thread(
            subprocess.run,
            command,
            cwd = str(AUTO_SCRIPT_PATH),
            capture_output = True,
            text = True,
            timeout = timeout_sec,
            check = False
        )
        duration_sec = round(time.perf_counter() - start, 2)

        return json.dumps({
            "status": "success" if result.returncode == 0 else "failed",
            "duration_in_seconds": duration_sec,
            "return_code": result.returncode,
            "working_directory": AUTO_SCRIPT_PATH.as_posix(),
            "command": command,
            "reports": _collect_report_paths(),
            "stdout_tail": __tail_output(result.stdout or "", max_output_chars),
            "stderr_tail": __tail_output(result.stderr or "", max_output_chars)

        })
    except subprocess.TimeoutExpired as e:
        duration_seconds = round(time.perf_counter() - start, 2)
        return json.dumps(
            {
                "status": "timeout",
                "error": f"Maven command exceeded timeout of {timeout_in_sec} seconds",
                "duration_seconds": duration_seconds,
                "working_directory": AUTO_SCRIPT_PATH.as_posix(),
                "command": command,
                "stdout_tail": __tail_output(__normalize_stream(e.stdout), max_output_chars),
                "stderr_tail": __tail_output(__normalize_stream(e.stderr), max_output_chars),
            },
            indent=2,
        )
    except Exception as e:
        return json.dumps(
            {
                "status": "error",
                "error": "Failed to execute Maven automation tests",
                "details": str(e),
                "working_directory": AUTO_SCRIPT_PATH.as_posix(),
            },
            indent=2,
        )

        
    


    


In [33]:
run_auto_script()

Resolved TestNG suite path: C:\Users\yashp\OneDrive\Documents\Phani\new_mcp\new_api_mcp\automation_mvn_tests\src\test\resources\testng.xml
In the _find_maven_command function


'{\n  "status": "error",\n  "error": "Maven command not found. Ensure Maven is installed and/or mvnw wrapper is present in the automation script directory."\n}'